# DocLens: parser arena, router and quantisation on Kaggle GPUs

Reproduces the GPU rows of the README: the same synthetic Japanese invoices (`eval/data/synth`, clean and scanned) go through
each parser, then through the same extraction step, and `eval/run_eval.py` writes one JSON per configuration into `eval/results/`.

**Settings:** Accelerator *GPU T4 x2*, Internet *on*. Run with *Save & Run All (Commit)* so it finishes headless.

**Nothing here is estimated.** Every number comes from a results file. The one assumption is the GPU price below, and it is
written into each result. Download `results.zip` from the last cell, unzip it into the repo, run `python eval/render_tables.py --write ../README.md`, commit.

| GPU | Used for |
|---|---|
| GPU 0 | the parser under test (Docling in-process, or a vLLM server for a VLM) |
| GPU 1 | the extraction LLM (vLLM) |

`--vram-device 0` samples `nvidia-smi` for GPU 0 when the parser is a vLLM server (torch can't see another process). That figure includes
vLLM's reserved KV cache; the weights-only figure is parsed from the vLLM log and stored in the result's notes.

In [ ]:
import json, os, pathlib, re, subprocess, time, urllib.request

REPO_URL = "https://github.com/shizen-57/PaperGraph.git"  # renamed repos redirect, so this keeps working
GPU_HOURLY_USD = "0.35"   # ASSUMPTION: replace with the hourly price of the GPU you want to quote. Recorded in every result.
os.environ["GPU_HOURLY_USD"] = GPU_HOURLY_USD
LLM = "Qwen/Qwen2.5-7B-Instruct-AWQ"        # extraction model for every arm
VLM = "Qwen/Qwen2.5-VL-7B-Instruct-AWQ"     # general VLM, prompted for Markdown

In [ ]:
!git clone $REPO_URL doclens_repo
%cd doclens_repo/doclens
!pip install -q -r requirements.txt docling vllm rapidfuzz apted lxml pytesseract faker
!apt-get install -y -q tesseract-ocr tesseract-ocr-jpn

In [ ]:
def start_vllm(model, port, gpu, *extra, max_len=8192, util=0.85):
    """OpenAI-compatible vLLM server pinned to one GPU. Returns the process; the log is vllm_<port>.log."""
    log = open(f"vllm_{port}.log", "w")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    p = subprocess.Popen(["python", "-m", "vllm.entrypoints.openai.api_server", "--model", model, "--dtype", "half",
                          "--max-model-len", str(max_len), "--gpu-memory-utilization", str(util), "--port", str(port), *extra],
                         stdout=log, stderr=log, env=env)
    for _ in range(180):
        try:
            urllib.request.urlopen(f"http://localhost:{port}/health")
            return p
        except Exception:
            time.sleep(10)
    raise RuntimeError(f"vLLM on :{port} did not start: see vllm_{port}.log")

def stop(p):
    p.terminate(); p.wait(); time.sleep(15)   # let the GPU memory free up

def weights_gib(port):
    m = re.search(r"Model loading took ([\d.]+) GiB", pathlib.Path(f"vllm_{port}.log").read_text())
    return f"weights {m.group(1)} GiB (from the vLLM log)" if m else None

def run_eval(name, parser, llm=LLM, variants="clean,scanned", vram=(), notes=(), env=None, reuse_parse=False):
    cmd = ["python", "eval/run_eval.py", "--parser", parser, "--extractor", "llm", "--llm", llm,
           "--variants", variants, "--out", f"eval/results/{name}.json"]
    for d in vram: cmd += ["--vram-device", str(d)]
    for n in notes: cmd += ["--note", n]
    if reuse_parse: cmd += ["--reuse-parse"]
    subprocess.run(cmd, check=True, env=dict(os.environ, LLM_BASE_URL="http://localhost:8001/v1", LLM_API_KEY="none", **(env or {})))

## Extraction LLM (GPU 1), shared by every arm below

In [ ]:
llm_server = start_vllm(LLM, 8001, 1, "--quantization", "awq")
llm_weights = weights_gib(8001)

## 1. Docling (GPU 0, in-process)
Docling's layout and table models run on the GPU when torch sees one. Peak VRAM here is `torch.cuda.max_memory_allocated` in the eval process.

In [ ]:
run_eval("docling-qwen2.5-7b", "docling", notes=[f"extractor: {LLM}, {llm_weights}"])

## 2. Classic baselines (CPU parse, same extractor)

In [ ]:
run_eval("tesseract-qwen2.5-7b", "tesseract")
run_eval("pdfium-qwen2.5-7b", "pdfium", variants="clean")   # a text layer cannot read scans; that is the point

## 3. VLMs as parsers (GPU 0, served by vLLM)
The VLM turns each page image into Markdown (`backend/vlm_parser.py`); the extraction step is identical to the other arms.
Check each model card for Japanese support and record it in REPORT.md.

In [ ]:
vlm_server = start_vllm(VLM, 8002, 0, "--quantization", "awq", "--limit-mm-per-prompt", "image=1")
run_eval("qwen-vl-qwen2.5-7b", "vlm", vram=[0], env={"VLM_BASE_URL": "http://localhost:8002/v1", "VLM_MODEL": VLM},
         notes=[f"parser: {VLM} zero-shot, prompt in backend/vlm_parser.py, {weights_gib(8002)}"])

In [ ]:
# Router (F2): Docling first, the VLM only on pages the signals send there. Needs the VLM server from the cell above.
run_eval("router-qwen2.5-7b", "router", vram=[0], env={"VLM_BASE_URL": "http://localhost:8002/v1", "VLM_MODEL": VLM},
         notes=["Docling in-process on GPU 0 + the VLM server on GPU 0; VRAM is the whole device"])
stop(vlm_server)

**PaddleOCR-VL (experimental).** This asks the model for whole-page OCR through vLLM with its task prompt. It is *not* the model's
layout-aware pipeline (PP-DocLayout + per-element prompts), so tables will be weaker than in the official pipeline: report it as such.
Check the model card for the current vLLM flags before running.

In [ ]:
PADDLE = "PaddlePaddle/PaddleOCR-VL"
paddle_server = start_vllm(PADDLE, 8002, 0, "--trust-remote-code", "--no-enable-prefix-caching", "--mm-processor-cache-gb", "0")
run_eval("paddleocr-vl-qwen2.5-7b", "vlm", vram=[0],
         env={"VLM_BASE_URL": "http://localhost:8002/v1", "VLM_MODEL": PADDLE, "VLM_PROMPT": "OCR:"},
         notes=[f"parser: {PADDLE}, whole-page 'OCR:' prompt, not the layout-aware pipeline, {weights_gib(8002)}"])
stop(paddle_server)

## 4. Quantisation (F7): FP16 vs 4-bit for the extraction model
Same parsed text (Docling output reused from step 1), same prompts; only the extraction model changes. Qwen2.5-3B is used because
the FP16 7B model (about 15 GB) does not fit one T4. Accuracy comes from the F1 / verified columns, speed from `extract.sec_per_doc`.

In [ ]:
stop(llm_server)
for tag, model, extra in [("fp16", "Qwen/Qwen2.5-3B-Instruct", []), ("awq", "Qwen/Qwen2.5-3B-Instruct-AWQ", ["--quantization", "awq"])]:
    server = start_vllm(model, 8001, 1, *extra, util=0.5)
    run_eval(f"quant-{tag}-qwen2.5-3b", "docling", llm=model, vram=[1], reuse_parse=True,
             notes=[f"quantisation comparison, {tag}: {weights_gib(8001)}"])
    stop(server)

## Collect

In [ ]:
!python eval/render_tables.py
!cd eval && zip -qr /kaggle/working/results.zip results